In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.fft as fft
from torch.utils.data import DataLoader, TensorDataset, Subset
import torchvision
import torchvision.transforms as transforms
from torchvision import models
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ================== Step 1: Load and Prepare CIFAR-10 Dataset ==================

def load_cifar10_dataset():
    """Load CIFAR-10 dataset with improved augmentation"""
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
    ])
    
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
    ])
    
    trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                            download=True, transform=transform_train)
    testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                           download=True, transform=transform_test)
    
    classes = ('plane', 'car', 'bird', 'cat', 'deer', 
               'dog', 'frog', 'horse', 'ship', 'truck')
    
    return trainset, testset, classes

# ================== Step 2: FFT Conversion Functions ==================

def spatial_to_frequency(images):
    """
    Convert spatial domain images to frequency domain using FFT
    Returns: frequency features (magnitude + phase encoding), phase, complex frequency
    """
    # Apply 2D FFT
    freq_complex = fft.fft2(images, dim=(-2, -1))
    freq_complex = fft.fftshift(freq_complex, dim=(-2, -1))
    
    # Extract magnitude and phase
    freq_magnitude = torch.abs(freq_complex)
    freq_phase = torch.angle(freq_complex)
    
    # Log-scale normalization for magnitude
    eps = torch.mean(freq_magnitude) * 0.01
    freq_magnitude_log = torch.log(freq_magnitude + eps)
    freq_magnitude_normalized = (freq_magnitude_log - freq_magnitude_log.mean()) / (freq_magnitude_log.std() + 1e-8)
    
    # Encode phase as cosine and sine
    phase_cos = torch.cos(freq_phase)
    phase_sin = torch.sin(freq_phase)
    
    # Concatenate magnitude and phase information (9 channels total)
    freq_features = torch.cat([freq_magnitude_normalized, phase_cos, phase_sin], dim=1)
    
    return freq_features, freq_phase, freq_complex

def frequency_to_spatial(freq_magnitude, freq_phase):
    """Convert frequency domain back to spatial domain"""
    # Reconstruct complex frequency representation
    freq_magnitude = torch.exp(freq_magnitude)
    freq_complex = freq_magnitude * torch.exp(1j * freq_phase)
    
    # Apply inverse FFT
    freq_complex = fft.ifftshift(freq_complex, dim=(-2, -1))
    spatial_complex = fft.ifft2(freq_complex, dim=(-2, -1))
    spatial_images = torch.real(spatial_complex)
    
    return spatial_images

# ================== Step 3: Frequency Domain Dataset ==================

class FrequencyDomainDataset(torch.utils.data.Dataset):
    """Custom dataset for frequency domain representations"""
    
    def __init__(self, original_dataset):
        self.original_dataset = original_dataset
        
    def __len__(self):
        return len(self.original_dataset)
    
    def __getitem__(self, idx):
        image, label = self.original_dataset[idx]
        
        # Convert to frequency domain
        freq_features, freq_phase, _ = spatial_to_frequency(image.unsqueeze(0))
        freq_features = freq_features.squeeze(0)
        freq_phase = freq_phase.squeeze(0)
        
        return freq_features, label, freq_phase

# ================== Step 4: CNN Model for Frequency Domain ==================

class FrequencyDomainCNN(nn.Module):
    """ResNet18-based model optimized for frequency domain (9-channel input)"""
    
    def __init__(self, base_model='resnet18', num_classes=10, dropout_rate=0.4):
        super(FrequencyDomainCNN, self).__init__()
        
        if base_model == 'resnet18':
            self.model = models.resnet18(pretrained=True)
            
            # Modify first conv layer for 9-channel input (3 mag + 3 cos + 3 sin)
            self.model.conv1 = nn.Conv2d(9, 64, kernel_size=3, stride=1, padding=1, bias=False)
            self.model.maxpool = nn.Identity()  # Remove maxpool for small 32x32 images
            
            # Enhanced classifier head
            self.model.fc = nn.Sequential(
                nn.Dropout(dropout_rate),
                nn.Linear(512, 256),
                nn.ReLU(inplace=True),
                nn.Dropout(dropout_rate * 0.5),
                nn.Linear(256, num_classes)
            )
        else:
            raise ValueError(f"Unsupported model: {base_model}")
        
        self._initialize_weights()
    
    def _initialize_weights(self):
        """Initialize new layers with proper weights"""
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        return self.model(x)
    
    def get_activations_gradient(self):
        """Hook for gradient extraction"""
        return self.gradients
    
    def activations_hook(self, grad):
        """Store gradients during backward pass"""
        self.gradients = grad
    
    def get_activations(self, x):
        """Extract feature maps from last conv layer before global pooling"""
        # Navigate through ResNet18 architecture
        x = self.model.conv1(x)
        x = self.model.bn1(x)
        x = self.model.relu(x)
        x = self.model.maxpool(x)
        
        x = self.model.layer1(x)
        x = self.model.layer2(x)
        x = self.model.layer3(x)
        x = self.model.layer4(x)  # Final conv features
        
        return x

# ================== Step 5: Training Functions ==================

class EarlyStopping:
    """Early stopping to prevent overfitting"""
    def __init__(self, patience=7, min_delta=0.0, verbose=True):
        self.patience = patience
        self.min_delta = min_delta
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.best_model_state = None
        
    def __call__(self, val_accuracy, model):
        score = val_accuracy
        
        if self.best_score is None:
            self.best_score = score
            self.best_model_state = model.state_dict()
        elif score < self.best_score + self.min_delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.best_model_state = model.state_dict()
            self.counter = 0

def train_model(model, train_loader, val_loader, epochs=30, lr=0.001, weight_decay=1e-4):
    """Train the frequency domain CNN model"""
    
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, 
        max_lr=lr * 10,
        epochs=epochs,
        steps_per_epoch=len(train_loader),
        pct_start=0.3,
        anneal_strategy='cos'
    )
    
    early_stopping = EarlyStopping(patience=15, min_delta=0.1, verbose=True)
    
    train_losses = []
    val_losses = []
    train_accuracies = []
    val_accuracies = []
    
    best_val_accuracy = 0.0
    best_model_state = None
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0
        
        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
        for i, (freq_images, labels, _) in enumerate(train_pbar):
            freq_images, labels = freq_images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(freq_images)
            loss = criterion(outputs, labels)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            scheduler.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()
            
            train_pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc': f'{100 * correct_train / total_train:.2f}%'
            })
        
        avg_train_loss = running_loss / len(train_loader)
        train_accuracy = 100 * correct_train / total_train
        train_losses.append(avg_train_loss)
        train_accuracies.append(train_accuracy)
        
        # Validation phase
        model.eval()
        running_val_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for freq_images, labels, _ in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]"):
                freq_images, labels = freq_images.to(device), labels.to(device)
                outputs = model(freq_images)
                loss = criterion(outputs, labels)
                running_val_loss += loss.item()
                
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        
        avg_val_loss = running_val_loss / len(val_loader)
        val_accuracy = 100 * correct / total
        val_losses.append(avg_val_loss)
        val_accuracies.append(val_accuracy)
        
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_model_state = model.state_dict()
        
        print(f'\nEpoch [{epoch+1}/{epochs}]')
        print(f'Train Loss: {avg_train_loss:.4f}, Train Acc: {train_accuracy:.2f}%')
        print(f'Val Loss: {avg_val_loss:.4f}, Val Acc: {val_accuracy:.2f}%')
        print(f'Learning Rate: {optimizer.param_groups[0]["lr"]:.6f}\n')
        
        early_stopping(val_accuracy, model)
        if early_stopping.early_stop:
            print("Early stopping triggered!")
            break
    
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"\nLoaded best model with validation accuracy: {best_val_accuracy:.2f}%")
    
    return train_losses, val_losses, train_accuracies, val_accuracies

# ================== Step 6: Score-CAM Implementation ==================

class ScoreCAM:
    """
    Score-CAM: Score-Weighted Visual Explanations for CNNs
    Applied to frequency domain images and mapped to spatial domain
    """
    
    def __init__(self, model, target_layer=None):
        self.model = model
        self.model.eval()
        self.activations = None
        self.target_layer = target_layer
        
    def get_activations(self, x):
        """Extract feature maps from the model"""
        return self.model.get_activations(x)
    
    def generate_cam(self, input_image, target_class, batch_size=16):
        """
        Generate Score-CAM for the input image
        Args:
            input_image: frequency domain image (1, 9, H, W)
            target_class: target class index
            batch_size: batch size for processing activation maps
        """
        # Get feature maps
        activations = self.get_activations(input_image)
        b, k, h, w = activations.shape
        
        # Get base prediction score
        with torch.no_grad():
            base_output = self.model(input_image)
            base_score = F.softmax(base_output, dim=1)[0, target_class].item()
        
        # Upsample activation maps to input size
        _, _, input_h, input_w = input_image.shape
        upsampled_activations = F.interpolate(
            activations, 
            size=(input_h, input_w), 
            mode='bilinear', 
            align_corners=False
        )
        
        # Normalize each activation map
        upsampled_activations = upsampled_activations.squeeze(0)  # (k, H, W)
        
        # Calculate weights for each activation map
        weights = []
        
        # Process in batches to save memory
        for i in range(0, k, batch_size):
            batch_end = min(i + batch_size, k)
            batch_activations = upsampled_activations[i:batch_end]
            
            batch_weights = []
            for act_map in batch_activations:
                # Normalize activation map to [0, 1]
                act_map_norm = act_map - act_map.min()
                if act_map_norm.max() > 0:
                    act_map_norm = act_map_norm / act_map_norm.max()
                
                # Create masked input
                masked_input = input_image * act_map_norm.unsqueeze(0).unsqueeze(0)
                
                # Get prediction score for masked input
                with torch.no_grad():
                    output = self.model(masked_input)
                    score = F.softmax(output, dim=1)[0, target_class].item()
                
                batch_weights.append(score)
            
            weights.extend(batch_weights)
        
        weights = torch.FloatTensor(weights).to(device)
        
        # Normalize weights
        if weights.max() > 0:
            weights = weights / weights.max()
        
        # Generate weighted CAM
        activations_2d = activations.squeeze(0)  # (k, h, w)
        cam = torch.zeros((h, w), dtype=torch.float32).to(device)
        
        for i, w in enumerate(weights):
            cam += w * activations_2d[i]
        
        # Apply ReLU to keep only positive influences
        cam = F.relu(cam)
        
        # Normalize CAM to [0, 1]
        if cam.max() > 0:
            cam = cam / cam.max()
        
        # Upsample to input size
        cam = F.interpolate(
            cam.unsqueeze(0).unsqueeze(0),
            size=(input_h, input_w),
            mode='bilinear',
            align_corners=False
        ).squeeze()
        
        return cam.cpu().detach().numpy(), weights.cpu().detach().numpy()

# ================== Step 7: Score-CAM to Spatial Domain Mapping ==================

def apply_scorecam_and_map_to_spatial(model, freq_image, phase, target_class, original_image):
    """
    Apply Score-CAM on frequency domain and map to spatial domain
    
    Args:
        model: trained frequency domain CNN
        freq_image: frequency domain image (1, 9, H, W)
        phase: phase information (1, 3, H, W) or (3, H, W)
        target_class: predicted class index
        original_image: original spatial domain image
    
    Returns:
        cam_spatial: CAM in spatial domain
        saliency_map: processed saliency map
        highlighted: highlighted original image
        original_np: original image as numpy array
    """
    
    model.eval()
    freq_input = freq_image.clone().detach().to(device)
    
    # Generate Score-CAM in frequency domain
    scorecam = ScoreCAM(model)
    cam_freq, weights = scorecam.generate_cam(freq_input, target_class, batch_size=32)
    
    # Extract magnitude component (first 3 channels)
    freq_magnitude = freq_input[:, :3, :, :].squeeze(0).cpu().detach()
    
    # Apply CAM as a mask to frequency magnitude
    cam_freq_tensor = torch.from_numpy(cam_freq).float()
    masked_freq_magnitude = freq_magnitude * cam_freq_tensor.unsqueeze(0)
    
    # Ensure phase has correct dimensions
    if phase.dim() == 4:
        phase = phase.squeeze(0)
    elif phase.dim() == 2:
        phase = phase.unsqueeze(0).repeat(3, 1, 1)
    
    # Convert back to spatial domain
    cam_spatial = frequency_to_spatial(
        masked_freq_magnitude.unsqueeze(0), 
        phase.unsqueeze(0)
    )
    cam_spatial = cam_spatial.squeeze(0)
    
    # Process spatial CAM
    cam_spatial = torch.abs(cam_spatial)
    saliency_map = torch.mean(cam_spatial, dim=0).numpy()
    
    # Apply Gaussian smoothing if available
    try:
        from scipy.ndimage import gaussian_filter
        saliency_map = gaussian_filter(saliency_map, sigma=1.5)
    except ImportError:
        pass
    
    # Normalize saliency map
    if saliency_map.max() > saliency_map.min():
        saliency_map = (saliency_map - saliency_map.min()) / (saliency_map.max() - saliency_map.min())
    else:
        saliency_map = np.zeros_like(saliency_map)
    
    # Apply threshold to highlight important regions
    threshold = np.percentile(saliency_map, 60)
    saliency_map = np.where(saliency_map > threshold, saliency_map, 0)
    
    # Re-normalize after thresholding
    if saliency_map.max() > 0:
        saliency_map = (saliency_map - saliency_map.min()) / (saliency_map.max() - saliency_map.min())
    
    # Denormalize original image
    if original_image.dim() == 4:
        original_image = original_image.squeeze(0)
    
    mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(3, 1, 1)
    std = torch.tensor([0.2023, 0.1994, 0.2010]).view(3, 1, 1)
    original_denorm = original_image.cpu() * std + mean
    original_denorm = torch.clamp(original_denorm, 0, 1)
    original_np = original_denorm.permute(1, 2, 0).numpy()
    
    # Create highlighted version with jet colormap
    saliency_colored = plt.cm.jet(saliency_map)[:, :, :3]
    alpha = 0.6 * saliency_map[:, :, np.newaxis]
    highlighted = (1 - alpha) * original_np + alpha * saliency_colored
    highlighted = np.clip(highlighted, 0, 1)
    
    return cam_spatial, saliency_map, highlighted, original_np, cam_freq

# ================== Step 8: Visualization Functions ==================

def plot_scorecam_results(original_np, freq_magnitude, cam_freq, saliency_map, 
                          highlighted, prediction, true_label, classes, confidence):
    """Plot comprehensive Score-CAM results"""
    
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    fig.suptitle('Frequency Domain CNN - Score-CAM Explainability Analysis', 
                 fontsize=16, fontweight='bold', y=0.995)
    
    # Row 1: Original pipeline
    axes[0, 0].imshow(original_np)
    axes[0, 0].set_title(f'Original Image\nGround Truth: {classes[true_label]}', 
                         fontsize=11, fontweight='bold')
    axes[0, 0].axis('off')
    
    freq_display = freq_magnitude[:, :3, :, :].squeeze(0).mean(0).cpu().numpy()
    im1 = axes[0, 1].imshow(freq_display, cmap='viridis')
    axes[0, 1].set_title('Frequency Domain\n(Magnitude Spectrum)', 
                         fontsize=11, fontweight='bold')
    axes[0, 1].axis('off')
    plt.colorbar(im1, ax=axes[0, 1], fraction=0.046, pad=0.04)
    
    im2 = axes[0, 2].imshow(cam_freq, cmap='jet')
    axes[0, 2].set_title('Score-CAM\n(Frequency Domain)', fontsize=11, fontweight='bold')
    axes[0, 2].axis('off')
    plt.colorbar(im2, ax=axes[0, 2], fraction=0.046, pad=0.04)
    
    correct = "✓" if prediction == true_label else "✗"
    color = 'green' if prediction == true_label else 'red'
    axes[0, 3].text(0.5, 0.5, f'{correct} Prediction:\n{classes[prediction]}\n\nConfidence:\n{confidence:.1f}%', 
                    ha='center', va='center', fontsize=13, fontweight='bold',
                    bbox=dict(boxstyle='round', facecolor=color, alpha=0.3))
    axes[0, 3].set_title('Model Prediction', fontsize=11, fontweight='bold')
    axes[0, 3].axis('off')
    
    # Row 2: Spatial domain results
    im3 = axes[1, 0].imshow(saliency_map, cmap='hot')
    axes[1, 0].set_title('Saliency Map\n(Mapped to Spatial)', fontsize=11, fontweight='bold')
    axes[1, 0].axis('off')
    plt.colorbar(im3, ax=axes[1, 0], fraction=0.046, pad=0.04)
    
    axes[1, 1].imshow(highlighted)
    axes[1, 1].set_title('Highlighted Regions\n(Overlay on Original)', 
                         fontsize=11, fontweight='bold')
    axes[1, 1].axis('off')
    
    axes[1, 2].imshow(original_np)
    axes[1, 2].imshow(saliency_map, cmap='jet', alpha=0.5)
    axes[1, 2].set_title('Importance Heatmap\n(50% Overlay)', fontsize=11, fontweight='bold')
    axes[1, 2].axis('off')
    
    # Side-by-side comparison
    axes[1, 3].imshow(np.concatenate([original_np, highlighted], axis=1))
    axes[1, 3].set_title('Before | After\n(Score-CAM Highlighting)', 
                         fontsize=11, fontweight='bold')
    axes[1, 3].axis('off')
    
    plt.tight_layout()
    plt.show()

def plot_training_curves(train_losses, val_losses, train_accuracies, val_accuracies):
    """Plot training and validation curves"""
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    epochs = range(1, len(train_losses) + 1)
    ax1.plot(epochs, train_losses, 'b-', label='Training Loss', linewidth=2)
    ax1.plot(epochs, val_losses, 'r-', label='Validation Loss', linewidth=2)
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Loss', fontsize=12)
    ax1.set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)
    
    ax2.plot(epochs, train_accuracies, 'b-', label='Training Accuracy', linewidth=2)
    ax2.plot(epochs, val_accuracies, 'r-', label='Validation Accuracy', linewidth=2)
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Accuracy (%)', fontsize=12)
    ax2.set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# ================== Step 9: Main Execution Pipeline ==================

def main():
    print("="*80)
    print("Frequency Domain CNN with Score-CAM Explainability Pipeline")
    print("="*80)
    
    print("\n[Step 1] Loading CIFAR-10 dataset...")
    trainset, testset, classes = load_cifar10_dataset()
    
    print("\n[Step 2] Splitting dataset into train/validation sets...")
    train_indices, val_indices = train_test_split(
        list(range(len(trainset))), 
        test_size=0.15,
        random_state=42,
        stratify=[trainset[i][1] for i in range(len(trainset))]
    )
    
    train_subset = Subset(trainset, train_indices)
    val_subset = Subset(trainset, val_indices)
    
    print(f"Training samples: {len(train_subset)}")
    print(f"Validation samples: {len(val_subset)}")
    print(f"Test samples: {len(testset)}")
    
    print("\n[Step 3] Converting to frequency domain using FFT...")
    freq_train_dataset = FrequencyDomainDataset(train_subset)
    freq_val_dataset = FrequencyDomainDataset(val_subset)
    freq_test_dataset = FrequencyDomainDataset(testset)
    
    train_loader = DataLoader(freq_train_dataset, batch_size=128, shuffle=True, 
                             num_workers=2, pin_memory=True)
    val_loader = DataLoader(freq_val_dataset, batch_size=128, shuffle=False,
                           num_workers=2, pin_memory=True)
    test_loader = DataLoader(freq_test_dataset, batch_size=1, shuffle=False)
    
    print("\n[Step 4] Initializing ResNet18 model for frequency domain...")
    model = FrequencyDomainCNN(base_model='resnet18', num_classes=10, dropout_rate=0.4).to(device)
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    print("\n[Step 5] Training model on frequency domain data...")
    train_losses, val_losses, train_accuracies, val_accuracies = train_model(
        model, train_loader, val_loader, 
        epochs=30,
        lr=0.001,
        weight_decay=5e-4
    )
    
    print("\n[Step 5.1] Plotting training curves...")
    plot_training_curves(train_losses, val_losses, train_accuracies, val_accuracies)
    
    print("\n[Step 6] Evaluating on test set...")
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for freq_images, labels, _ in tqdm(test_loader, desc="Testing"):
            freq_images, labels = freq_images.to(device), labels.to(device)
            outputs = model(freq_images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    test_accuracy = 100 * correct / total
    print(f"\nFinal Test Accuracy: {test_accuracy:.2f}%")
    
    print("\n[Step 7] Applying Score-CAM and mapping to spatial domain...")
    print("Generating explainability visualizations for sample images...\n")
    
    # Test on multiple samples
    test_indices = [0, 10, 20, 30, 40]
    
    for idx in test_indices:
        # Get original image
        original_image, true_label = testset[idx]
        
        # Convert to frequency domain
        freq_features, phase, _ = spatial_to_frequency(original_image.unsqueeze(0))
        freq_input = freq_features.to(device)
        
        # Get model prediction
        model.eval()
        with torch.no_grad():
            output = model(freq_input)
            probabilities = F.softmax(output, dim=1)
            confidence, predicted = torch.max(probabilities.data, 1)
            predicted_class = predicted.item()
            confidence = confidence.item() * 100
        
        print(f"Sample {idx}:")
        print(f"  True Label: {classes[true_label]}")
        print(f"  Predicted: {classes[predicted_class]} ({confidence:.1f}% confidence)")
        
        # Apply Score-CAM and map to spatial domain
        cam_spatial, saliency_map, highlighted, original_np, cam_freq = apply_scorecam_and_map_to_spatial(
            model, freq_input, phase.squeeze(0), predicted_class, original_image
        )
        
        # Visualize results
        plot_scorecam_results(
            original_np,
            freq_features, 
            cam_freq,
            saliency_map, 
            highlighted,
            predicted_class, 
            true_label, 
            classes,
            confidence
        )
        
        # Clear GPU cache
        torch.cuda.empty_cache()
        print()
    
    print("="*80)
    print("Pipeline completed successfully!")
    print(f"Final Test Accuracy: {test_accuracy:.2f}%")
    print("="*80)
    print("\nKey Features Implemented:")
    print("✓ FFT-based spatial to frequency domain conversion")
    print("✓ ResNet18 model trained on frequency domain data")
    print("✓ Score-CAM explainability in frequency domain")
    print("✓ Inverse FFT mapping to spatial domain")
    print("✓ Highlighted important regions in original images")
    print("="*80)

if __name__ == "__main__":
    main()